# Chapter 8 — Template Method Pattern
## *Encapsulating Algorithms*

**Intent:** Define the **skeleton of an algorithm** in a base class, deferring some steps to subclasses.  
Template Method lets subclasses redefine certain steps without changing the algorithm's structure.

### OO Design Principle
- **Hollywood Principle:** "Don't call us, we'll call you."  
  The base class calls the subclass methods, not the other way around.

### Key concepts
- **Abstract steps** — must be implemented by subclasses.
- **Concrete steps** — shared implementation in the base class.
- **Hooks** — optional steps with a default (empty) implementation; subclasses may override.

### Book context
Making tea and coffee follow the same recipe skeleton but differ in steeping/brewing and add-ins.

In [ ]:
from abc import ABC, abstractmethod

class CaffeineBeverage(ABC):
    # Template method — final algorithm skeleton
    def prepare_recipe(self):
        self.boil_water()
        self.brew()                  # abstract — subclass decides
        self.pour_in_cup()
        if self.customer_wants_condiments():  # hook
            self.add_condiments()    # abstract

    # Concrete steps shared by all beverages
    def boil_water(self):   print("Boiling water")
    def pour_in_cup(self):  print("Pouring into cup")

    # Abstract steps — subclasses must implement
    @abstractmethod
    def brew(self): ...
    @abstractmethod
    def add_condiments(self): ...

    # Hook — optional override
    def customer_wants_condiments(self) -> bool:
        return True


class Tea(CaffeineBeverage):
    def brew(self):           print("Steeping the tea")
    def add_condiments(self): print("Adding lemon")


class Coffee(CaffeineBeverage):
    def brew(self):           print("Dripping coffee through filter")
    def add_condiments(self): print("Adding sugar and milk")


class BlackCoffee(CaffeineBeverage):
    def brew(self):           print("Dripping coffee through filter")
    def add_condiments(self): pass  # never called because hook returns False
    def customer_wants_condiments(self): return False


print("--- Making Tea ---")
Tea().prepare_recipe()
print("\n--- Making Coffee ---")
Coffee().prepare_recipe()
print("\n--- Making Black Coffee (no condiments) ---")
BlackCoffee().prepare_recipe()

## Real-world example — data processing pipeline

In [ ]:
import csv, json
from io import StringIO

class DataProcessor(ABC):
    # Template method
    def process(self, raw: str) -> list[dict]:
        parsed  = self.parse(raw)
        cleaned = self.clean(parsed)
        if self.should_validate():
            self.validate(cleaned)
        return cleaned

    @abstractmethod
    def parse(self, raw: str) -> list[dict]: ...

    def clean(self, data: list[dict]) -> list[dict]:
        return [{k: v.strip() if isinstance(v, str) else v
                 for k, v in row.items()} for row in data]

    def validate(self, data: list[dict]):
        for row in data:
            assert row, "Empty row found"

    def should_validate(self) -> bool:
        return True


class CsvProcessor(DataProcessor):
    def parse(self, raw: str) -> list[dict]:
        return list(csv.DictReader(StringIO(raw)))


class JsonProcessor(DataProcessor):
    def parse(self, raw: str) -> list[dict]:
        return json.loads(raw)

    def should_validate(self): return False  # trust JSON schema


csv_data = "name,age\nAlice , 30\n Bob,25"
json_data = '[{"name": "Alice", "age": 30}]'

print(CsvProcessor().process(csv_data))
print(JsonProcessor().process(json_data))

## Python's `unittest.TestCase` uses Template Method

`setUp()`, `tearDown()`, `setUpClass()` are hooks.  
The test runner calls `run()` which is the template method.

In [ ]:
import unittest

class MyTest(unittest.TestCase):
    def setUp(self):     print("  setUp hook")
    def tearDown(self):  print("  tearDown hook")

    def test_example(self):
        print("  running test")
        self.assertEqual(1 + 1, 2)

# Run programmatically
suite = unittest.TestLoader().loadTestsFromTestCase(MyTest)
unittest.TextTestRunner(verbosity=2).run(suite)

## Interview cheat-sheet

| Question | Answer |
|---|---|
| Template Method vs Strategy? | TM uses inheritance; the algorithm skeleton lives in base class. Strategy uses composition; the whole algorithm is swapped. |
| What is a hook? | A method with a default (often empty) implementation that subclasses *may* override to plug into the algorithm. |
| Hollywood Principle? | Base class controls when to call subclass methods — "don't call us, we'll call you." |
| Python examples? | `unittest.TestCase`, `http.server.BaseHTTPRequestHandler`, `cmd.Cmd`. |

**Pattern family:** Behavioral